# Machine learning assignment, Theodor Helje

## Introduction
---

The goal of this assignment was to develop a recommendation system that reccomends movies based on an input movie. This was to be based on the movielens data set *ml-latest*.

This report aims to explain the choises made during the development of the reccomendation system as well as what methods were used and what limitations apply as a consequence of those methods.

## Method
---

### Data

The data used was acquired through grouplens.org and contains information regarding movies, users and the interaction between users and movies in the form of tags and ratings. The data is gathered from the web site movielens.org over various periods of time. This data set contains various files, of which only movies.csv, ratings.csv and tags.csv are used. These three individual data sets were chosen due to them collectively containing all essential data needed to create a hybrid (collaborative filtering and content based filtering) reccomendation system.

The movies.csv data set contains 86,537 rows, each corresponding to a movie with the columns containing movie id, title and genres.

The ratings.csv data set contains 33,832,160 rows, each corresponding to an individual rating with columns describing user id, movie id, rating and timestamp. Timestamp is not used in the model since it is deemed to be irrelevant to the project.

The tags.csv data set contains 2,328,315 rows, each corresponding to a custom tag applied by a user. These tags work as comments and are not necessarily relevant or correct in the context of the movies. For each row (tag), the columns describe user id, movie id, tag and timestamp. Timestap is yet again deemed irrelevant.

### Methodology

The recommendation systen was built by combining and weighting both content-based filtering and collaborative filtering, creating a hybrid system. The goal of this solution is to reduce both individual reccomendation models weaknesses while amplifying their strengths. While a content-based model would define a movie as its features (such as genres) and a collaborative model would define it as the users "taste profiles" that interact with it (in this case through ratings), a hybrid model would define a movie as a combination of its attributes and users interactions with it. This leads to a model that captures meaning in both movie descriptions and trends in user behavure.

The content-based component of the model consists of a movie feature matrix. This matrix is of shape (n_movies, m_features) with each movie being represented as a normalized row vector consisting of two weighted normalized vectors, one containing one-hot encoded genres and the other containing TF-IDF encoded tags. This implementation allows for more nuance by normalizing both the genres-vector and the TF-IDF-vector separately before weighting them and adding them togeather, effectively removing the otherwise heavy bias towards genres in relation to tags.

The collaborative filtering component of the model consists of a user interaction matrix of the shape (n_users, m_movies). In this matrix, each entry is the standardized (in relation to all ratings) value of the rating the corresponding user (row index mapped) gave the corresponding movie (column index mapped).

Both of these matrices are saved and operated on as sparse (csr) matrices due to both their sparse nature and their relatively big size. These matrices only save non-zero values, allowing the program to run on 8 GB ram as opposed to normal matrices with every value saved, which for only the collaborative filtering matrix would take up around 26 GB.

Cosine similarity was used to calculate the similatiry used for recommendations. This calculated the cosine of the angle between two vectors, resulting in a value ranging from -1 to 1 representing similarity. Due to the structuring of the data, cosine similarity was a clear choise since both movies and users are easily represented as normalized vectors of fixed dimensions.

### Implementation

When implementing these methods and developing the system, the code was split into 5 files: data, preprocessing, models, pipeline and main. This was done so that the code would be both clear and easy to follow. 

The loading and saving of the data is handled in the data.py file. This file also contains the functions for saving and checking the state of the system. All data from the data set is initially loaded using pandas due to the ease of use.

The file preprocessing.py handles all of the feature engineering and data restructuring with the exeption of creating the final embeddings, which is done in models.py. This file handles the entire process needed to transform the raw input data into the needed matrices for the creation of the final embeddings. This file also handles tasks such as finding movie ids based on an input title and creating the mapping dicts used throughout the program. The general process this file is responsible for is one-hot encoding movies, TF-IDF encoding tags, using these to create a movie feature matrix and creating a user interaction matrix using ratings.

The models are all handled in the models.py file. This also includes the creation of the embeddings used in the model, which is placed in this file due to the TruncatedSVD being used to do this. This files first function is get_embeddings(), which is used to transform the prevously explained matrices to usable embeddings of reduced dimensionality. It does this both for the user interaction matrix, creating a collaborative movie embedding, and for the movie feature matrix, creating a content based embedding. These are then weighted and combined using hstack to create the movie embeddings matrix used when calculating similarity between movies. This function also creates an embedded user matrix where vectors correspond with user preferences. This is done using matrix multiplication by multiplying the user interaction matrix with the movie embeddings matrix. Both of these embedding matrices are then normalized.

The other two functions in models.py calculate the actual movie reccomendations, one based on a movie and the other one based on a user. These work similarly in that they both use cosine similatiy to calculate "scores" for each movie representing how alike it is to the input preference or movie. They then return the movie ids of the n most alike movies as reccomendations. The main difference in the functions is that, although both cosine similarity calculations can be expressed as A x B where A is the movie embeddings matrix, B differes in the two functions by either taking the form of a movie vector embedding or a user preference vector embedding. In the case of finding movies based on user preferences, previously rated movies are set to -infinity in the scores to insure they are never reccomended.

The pipeline.py file is then used to orchestrate all of the functions in data.py, preprocessing.py and models.py such that the process of using the reccomendation system is much less of a manual process. This means that pipeline.py contains functions such as model_setup(), which both validates saved files and checks saved state, and predict_movie_reccomendations(), which makes using the actual reccomendation system as a whole a much easier process.

Finally, main.py is designed as "a layer above pipeline.py". This means that it is simply designed as the outer layer for running the reccomendation system, in this case only in the terminal. The file contains two functions, one to run the movie reccomender in the terminal and the other one to run the user reccomender. This means that, if this system is to be imported and used as a part of a bigger project, main.py would not need to be imported as it essentially is only a minimalistic interpreting layer between the console and the system.

## Result
---

In [43]:
from pipeline import model_setup, predict_movie_reccomendations

In [44]:
model_setup()
predict_movie_reccomendations("yesterday")

FileNotFoundError: file Labb-1/hyperparameters.yml not found